# IN4080: obligatory assignment 3
 
Mandatory assignment 3 is about the practical use of Large Language Models (LLMs). More specifically, you will be tasked to implement a RAG (Retrieval-Augmented Generation) system able to answer factual questions based on a document database, more specifically Wiki pages extracted from an [online Star Wars encyclopedia](https://starwars.fandom.com). 

You are required to get at least 12/20 points to pass. 

- We assume that you have read and are familiar with IFI’s requirements and guidelines for mandatory assignments, see [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-mandatory.html) and [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-guidelines.html).
- This is an individual assignment. You should not deliver joint submissions. 
- You may redeliver in Devilry before the deadline (__Friday, October 23 at 23:59__).
- Only the last delivery will be read! If you deliver more than one file, put them into a zip-archive. You don't have to include in your delivery the data files already provided for this assignment. 
- Name your submission _your\_username\_in4080\_mandatory\_3_


<div class="alert alert-block alert-success"> <b>Practical setup:</b> The preferred way to complete this assignment is using the high-performance computing cluster <i>Fox</i>. See <a href="https://www.uio.no/studier/emner/matnat/ifi/IN4080/h25/computing-setup.html">here</a> for instructions on how to register and log in to Fox. 

Note: if you cannot work on Fox, you can do the assignment on your machine. In this case, make sure you have installed the following libraries: `pytorch`, `transformers`, `huggingface_hub`,  `bm25s`, `tf_keras`, `sentence-transformers` and either `nltk` or `spacy` (depending on how you prefer to do sentence splitting).

As the Gemma LLMs (from Google) are <a href="https://huggingface.co/docs/hub/models-gated">gated</a>, you will also need to:

- Create a user account on HuggingFace
- Access the user conditions for <a href="https://huggingface.co/google/gemma-3-1b-it">Gemma 3</a>
- Create an access token (<a href="https://huggingface.co/settings/tokens">here</a>)
- At the start of your notebook, run the following code:
```python
from huggingface_hub import login
login(token="hf_...")
```

</div>

You should deliver a completed version of this Jupyter notebook, containing both your code and explanations about the steps you followed. We want to stress that simply submitting code is __not__ by itself sufficient to complete the assignment - we expect the notebook to also contain explanations of what you have implemented, along with motivations for the choices you made along the way. Preferably use whole sentences, and mathematical formulas if necessary. Explaining in your own words (using concepts we have covered through in the lectures) what you have implemented and reflecting on your solution is an important part of the learning process - take it seriously!


Regarding the use of LLMs (ChatGPT or similar): you are allowed to use them as 'sparring partner', for instance to clarify something you have not understood. However, you are __not__ allowed to use them to generate solutions (either in part or in full) to the assignment tasks. 

--------------------
## Before starting to read my code and results, I tried to provide informative comments for the code, and for each part that was needed, I wrote a Markdown like the following.

--------------------
### My explanation
You can find my "blablabla" here :)

--------------------

## Thanks for your attention

## Basic setup

We will start by building a chatbot that directly answers user questions using an instruction-tuned LLM, without relying on any database. We will use the instruction-tuned version of the Gemma 3 language model from Google, which is available on HuggingFace. 

_Note: feel free to switch to another model (such as the Llama, Qwen, Deepseek or Mistral models) if you wish to experiment with them._



**Task 1** (4 points): Drawing inspiration from the code examples on the [Gemma webpage](https://huggingface.co/google/gemma-3-1b-it), implement the `__init__` and `get_response` methods. If you run the code on Fox with a GPU (or on a personal machine with a GPU), make sure that your code actually runs on the GPU. Do not forget to use our `cache_dir` for the course, where we have already downloaded the models for you. You should use `cache_dir` when initializing/loading the model.

In [ ]:
from huggingface_hub import login
# login()
import warnings
warnings.filterwarnings("ignore")

--------------------
### My explanation

One thing that I should mention: I didn't use the cache because, in the cluster that I am using, I haven't had access to the cache dir.


--------------------

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#Define the device to send the model to, either GPU or CPU depending on availability
device = "cuda" if torch.cuda.is_available() else "cpu"

class BasicResponseGenerator:

    def __init__(
        self,
        model_name="google/gemma-3-1b-it",
        # cache_dir="/fp/projects01/ec403/hf_models",
        device=device,
        
    ):
        """Loads the tokenizer and pretrained causal LM for the given model.
        If a GPU is available, the model should be loaded on the GPU"""

        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        # self.quantization_config = BitsAndBytesConfig(load_in_8bit=True)
        # self.model = AutoModelForCausalLM.from_pretrained(model_name,quantization_config=self.quantization_config).eval()
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).eval()
        self.model.to(self.device)

    # Here I normalize the output of the model.
    # For example, remove the meta information, and only return the text of the response.
    def output_normalization(self, output):
        # Remove the question from the output, and only return the answer.
        
        response = output.split('<start_of_turn>model')[1].strip().split('<end_of_turn>')[0].strip()
        return response

    def get_response(self, prompt: str, max_length: int = 50, rule_prompt: str = None) -> str:
        """Given a prompt, generate a response (of a maximum max_length tokens) and return it.
        Only the response should be returned, not the text of the prompt itself
        """

        # First we define a template for the conversation. The template is a list of messages, where each message is a dictionary with a role and content. 
        # The role can be "system", "user", or "assistant". 
        # The content is a list of dictionaries, where each dictionary has a type and text. The type can be "text" or "image". 
        # The text is the actual text of the message.

        if rule_prompt is None:
            rule_prompt = "You are a helpful assistant a fan of the world of Star Wars. Just give the straightforward answer."
        
        messages = [
            [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": "{rule}".format(rule=rule_prompt)},]
                },
                {
                    "role": "user",
                    "content": [{"type": "text", "text": "{prompt}".format(prompt=prompt)},]
                },
            ],
        ]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)


        with torch.inference_mode():
            outputs = self.model.generate(**inputs, max_new_tokens=max_length)

        outputs = self.tokenizer.batch_decode(outputs)
        return self.output_normalization(outputs[0])


--------------------
### My explanation

1. I load the model in bfloat16. It needs half the memory of float32, and the quality is almost the same.
2. I use the chat template of the tokenizer (apply_chat_template), because Gemma is an instruction-tuned model and it expects its special turn tokens. I also added a system message telling the model it is a Star Wars assistant and should give a straightforward answer, so the answers stay short.
3. Generation runs inside torch.inference_mode(), because we don't need gradients and it saves memory and time.
4. max_length is passed as max_new_tokens, so it limits only the generated part, not the prompt.
5. The decoded output contains the whole conversation (prompt + answer + special tokens). So in output_normalization I split on <start_of_turn>model and <end_of_turn> and only keep the text of the model's answer.

--------------------

In [19]:
agent = BasicResponseGenerator()

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

_Note: An easy way to verify that the GPU is actually used is to run the command `nvidia-smi` while your code is running. There also exists alternative GPU monitoring tools, like [`gpustat`](https://pypi.org/project/gpustat/0.3.2/)._

You can then test your response generator with the following set of questions: 

In [21]:
# If it shows cuda, it means we are using the gpu
print(agent.model.device)

cuda:0


In [3]:
questions = [
    "Who is Luke Skywalker?",
    "Where is the Niima Outpost in Star Wars?",
    "Have you heard of Nute Gunray? Who is he?",
    "What kind of planet is Kashyyyk, and who discovered it?",
    "Who are Condlurans, and can you give 2-3 names of known Condlurans?",
    "What can you tell me about the First Battle of Geonosis?",
    "What is the name of the settlement where Anakin Skywalker and his mother lived?",
    "Which planet did Darth Sidious represent as senator?",
]

In [ ]:


for question in questions:
    print("Question:", question)
    print("Answer:", agent.get_response(question))
    print("-------")

Question: Who is Luke Skywalker?
Answer: Luke Skywalker is a young farm boy who becomes a Jedi Knight and fights against the Empire.
-------
Question: Where is the Niima Outpost in Star Wars?
Answer: It’s currently located in the Outer Rim Territories, specifically on the planet **Nima**.
-------
Question: Have you heard of Nute Gunray? Who is he?
Answer: Yes, I have! Nute Gunray is a really important and complex character in Star Wars. He was a powerful and ruthless Imperial officer, notorious for his brutal tactics and his dark secret – he was a former Jedi Knight who fell to the dark
-------
Question: What kind of planet is Kashyyyk, and who discovered it?
Answer: Kashyyyk is a lush, temperate planet with a unique, almost ethereal beauty. It was discovered by the Rebel Alliance during the Battle of Endor.
-------
Question: Who are Condlurans, and can you give 2-3 names of known Condlurans?
Answer: Condlurans are a species of reptilian humanoids from the planet Xylos, known for their

--------------------
### My explanation
Yes, as you mentioned in the following some answers are incorrect, for example, I repeated the following question and received wrong answer in each time

Question: Which planet did Darth Sidious represent as senator?

Answer: Galactic Senate.
or: Coruscant.

Which are not true, because the Galactic Senate is not a planet and is is the institution he served in and the name of panet is Naboo :)

--------------------

If your implementation is correct, the model should give you a few correct answers, but also many responses for which the model is either unable to give a precise answer, or hallucinates a (wrong) answer. This is expected, as the model is relatively small (one billion parameters) and is a generic model that is not particularly optimised to generate trivia about the Star Wars Franchise. We will now try to improve the model performance by coupling the LLM to a document database.

## Retrieval step

Retrieval-augmented generation operates on a simple idea: instead of directly generating a response based on the "parametric knowledge" of the LLM, we first search for relevant documents in a database (or on the web). We then include the most relevant documents to the prompt, and ask the LLM to answer the user question _based on this retrieved knowledge_. 

In this assignment, you will use a set of Wiki texts extracted from an [online Star Wars encyclopedia](https://starwars.fandom.com) as document database. The wiki texts are available as a JSON file [here](https://home.nr.no/~plison/data/starwars.json). The JSON is simply a dictionary mapping Wiki page titles to their content (in plain text).

### Sparse retrieval 

We can start by using the [BM25s](https://bm25s.github.io/) library, which implements a number of well-known search algorithms, which are all variants of the original [BM25 algorithm](https://en.wikipedia.org/wiki/Okapi_BM25) . Although BM25 is an old-fashioned search technique based on bag-of-words, it remains suprisingly effective, and is still widely used in modern NLP systems.

**Task 2** (4 points): Fill in the implementation for the `BM25Retriever` class using [BM25s](https://bm25s.github.io/) (see the library documentation for details). You should filter out stop words by adding `stopwords='en_plus'` to the arguments of the tokenizer. 

In [ ]:
# I used the following command to download the Star Wars dataset for testing:
# !wget https://home.nr.no/~plison/data/starwars.json

--2026-10-17 13:22:13--  https://home.nr.no/~plison/data/starwars.json
Connecting to 10.110.0.12:3128... connected.
Proxy request sent, awaiting response... 200 OK
Length: 67629498 (64M) [application/json]
Saving to: ‘starwars.json’

starwars.json       100%[===================>]  64.50M   368MB/s    in 0.2s    

2026-10-17 13:22:14 (368 MB/s) - ‘starwars.json’ saved [67629498/67629498]



In [4]:
import json
with open("starwars.json") as f:
    starwars_data = json.load(f)
    print("Number of entries in the Star Wars dataset:", len(starwars_data))
    print("First entry in the Star Wars dataset:", list(starwars_data.items())[0])

Number of entries in the Star Wars dataset: 178246
First entry in the Star Wars dataset: ('Brianna', 'Brianna, also known as "The Last Handmaiden," was a half-Human, half-Echani hybrid fighter of Jedi heritage. The illegitimate daughter to the famous Echani General Yusanis and Jedi Master Arren Kae, both of whom fought in the Mandalorian Wars, she later served along with her five half-sisters as the Handmaiden Sisters under Jedi Master Atris in the period following the Jedi Civil War. During her service to Atris at the Jedi Academy on Telos IV, Brianna met Meetra Surik, also known as the Jedi Exile, and was subsequently tasked with accompanying her on her mission to seek out the Lost Jedi, Jedi Masters who survived the First Jedi Purge. As Surik\'s companion, Brianna helped battle against numerous adversaries, including the Sith Triumvirate.\n\nDuring the journey, Brianna trained Surik in Echani techniques and eventually received Jedi training from Surik in return, becoming one of the 

In [5]:
import bm25s
import json
from typing import List


class BM25Retriever:

    def __init__(self, filename="starwars.json"):
        """Using the json file provided as input, create a BM25s retriever
        containing all (indexed) documents."""

        # First we should load the json file
        with open(filename, "r") as f:
            starwars_data = json.loads(f.read())
            # Then we should construct the corpus from the json data. 
            # Each element of the corpus is the key which is a phrase related to Star Wars, and the value is an explanation related to that phrase.
            self.corpus = [
                f"{key}: {value}" for key, value in starwars_data.items()
            ]

        self.corpus_tokens = bm25s.tokenize(self.corpus, stopwords="en_plus")
        self.retriever = bm25s.BM25(corpus=self.corpus)
        self.retriever.index(self.corpus_tokens)

    def search(self, query: str, k: int = 5, cal_scores: bool = False) -> List[str]:
        """Use the BM25 retriever to find the k documents that are closest
        to the provided query"""

        query_tokens = bm25s.tokenize(query, stopwords="en_plus")
        docs, scores = self.retriever.retrieve(query_tokens, k=k)
    
        if cal_scores:
            return docs[0].tolist(), scores
        else:
            return docs[0].tolist()


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


We can then test our retriever by checking whether the documents with highest BM25 scores are indeed the ones that are most relevant to the query:

In [13]:
retriever = BM25Retriever()
for question in questions:
    print("Question:", question)
    print("Retrieved documents:")
    for relevant_doc in retriever.search(question):
        print("- " + relevant_doc.replace("\n", " "))
    print("===========")

Split strings:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/178246 [00:00<?, ?it/s]

Question: Who is Luke Skywalker?
Retrieved documents:


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

- Finn's mother: Finn's mother lived with him on Coruscant. In 32 BBY, she helped Anakin Skywalker return to the Jedi Temple after he helped her son repair their malfunctioning nanny droid.
- Mos Espa: Mos Espa was a spaceport settlement located on the desert world of Tatooine. The settlement included a number of commercial and workspace settings, as well as entertainment establishments such as the Mos Espa Grand Arena. During the Invasion of Naboo, Mos Espa was also home to a number of slaves, including Anakin Skywalker and his mother, Shmi.
- Skywalker home: The Skywalker home was where the slaves Anakin Skywalker and his mother Shmi Skywalker Lars lived in Mos Espa on the desert planet Tatooine. By the time the handmaiden Sabé traveled to Tatooine on behalf of Padmé Amidala to free Shmi from slavery, Shmi no longer lived in the home and the door had a newly cut symbol of a white sun on it.Queen's Shadow
- Tusken Raider camp (Anakin Skywalker): A Tusken Raider camp was located in the

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

- Dawk: Dawk was a planet in the Outer Rim Territories that was orbited by a moon. During the Republic Era, between around 40 BBY and 32 BBY,The events of Darth Maul &ndash; Black, White & Red 3 take place during Darth Maul's time as the Sith apprentice of Darth Sidious, which Star Wars: Timelines dates to between around 40 BBY and 32 BBY. the Devaronian criminal Coir Cion operated on the moon of Dawk. The Sith Lord Darth Sidious dispatched his Sith apprentice, Darth Maul, to kill Cion and his associates after the Devaronian threatened to blackmail Senator Sheev Palpatine&mdash;Sidious's public persona&mdash;with information on his underworld dealings.
- Darth Sidious' shuttle: Darth Sidious' shuttle was a Republic executive shuttle used as the personal shuttle of the Sith Lord Darth Sidious in the later stages of the Clone Wars. He used the ship to travel to the planet Mandalore to thwart his former apprentice Maul's rise to power.
- Massacre at the Gran Protectorate Embassy: The mass

--------------------
### My explanation

One thing:

I think one drawback of BM25 is that it treats the words of the query independently, without understanding how they relate to each other.
For example, again in the last question, what matters is the *combination* of "Darth Sidious", "planet" and "senator", since together they ask a specific question. BM25, however, just scores each word on its own and sums the scores.
As a result, it returns documents that contain these words (e.g., planets where Sidious was involved), but none of them actually answers the question.

--------------------

If your implementation is correct, the retrieved documents should for the most part relevant to the query. 

### Dense retrieval 

Many of those documents are, however, way too long to be included in the prompt for our Gemma model (especially if we wish to include 4-5 retrieved texts for each query!). Can we ensure that the length of each retrieved text stays within a reasonable length, such as one or two sentences? 

One strategy is to not return the full documents, but instead determine the most relevant _sentences_ within those documents. But how do we determine which sentence is most relevant? A sparse retriever using BM25 would not work well here, as it does not really account for the semantics of the query. Instead, what we can do is to:
- split the documents (retrieved through BM25) into sentences
- extract sentence embeddings for the query and for each sentence
- compute the cosine similarities between the query vector and each sentence vector
- and return the _k_ most similar sentences

In other words, our approach starts with a _sparse retrieval step_ at the level of full documents (which we already have implemented, using BM25S), and continues with a _dense retrieval step_ to determine the most relevant sentences among the sentences that are found in the retrieved documents.

**Task 3** (4 points): Re-implement the `search` method to segment into sentences each document retrieved with BM25, extract sentence embeddings for the query and sentences using the encoder model (see [here](https://sbert.net/examples/applications/semantic-search/README.html) for explanations and code examples), and then select the _k_ sentences with highest cosine similarities. Here, we are also loading the encoder model from our course `cache_dir`.

_Tips_: You can use `nltk.sent_tokenize` to segment your document in sentences. If you have problems with `nltk`, you can also use the sentence splitter from `spacy`, or anything else that works for you.

--------------------
### My explanation

The hybrid retriever has two steps:

1. **Sparse step:** first I call the BM25 search from the parent class to get the k best documents. This step is fast and reduces 178k documents to only a few.
2. **Dense step:** then I split these documents into sentences with nltk.sent_tokenize, and encode the query and all sentences with all-MiniLM-L6-v2. I compute the cosine similarity between the query vector and each sentence vector.

and return the k sentences with the highest similarity (torch.topk). I use min(k, len(sentences)) in case the documents have fewer sentences than k.

The point is that BM25 only matches words, but the encoder can understand the meaning of the question. So among the sentences of the retrieved docs, it can choose the ones that actually answer it, and the context we give to Gemma becomes much shorter.

--------------------

In [6]:
import nltk
import bm25s
import re, json
import numpy as np
import sentence_transformers

# nltk.download('punkt_tab')
from typing import List


class HybridRetriever(BM25Retriever):

    def __init__(
        self,
        filename="starwars.json",
        encoder_model="all-MiniLM-L6-v2",
        cache_dir="/fp/projects01/ec403/hf_models",
    ):
        """Using the json file provided as input, create a BM25s retriever
        containing all (indexed) documents, and loads a sentence transformer model
        used to compute the embeddings for the query and sentences"""

        BM25Retriever.__init__(self, filename)
        self.encoder = sentence_transformers.SentenceTransformer(
            encoder_model, 
        )

    def search(self, query: str, k: int = 5) -> List[str]:
        """Use the BM25 retriever to find the documents that are closest
        to the provided query, and then the sentence transformer model to
        determine the most relevant sentences"""

        # First sparse retrieval 
        docs = BM25Retriever.search(self, query, k)
        # Then, convert the docs to sentences
        sentences = [sentence for doc in docs for sentence in nltk.sent_tokenize(doc)]
        # Afterwards, we compute the embeddings for the query and sentences, and then compute the similarity between the query and sentences. 
        query_emb = self.encoder.encode_query(query, convert_to_tensor=True)
        sentences_emb = self.encoder.encode_document(sentences, convert_to_tensor=True)

        similarities = self.encoder.similarity(query_emb, sentences_emb)[0]
        scores, idx = torch.topk(similarities, k=min(k, len(sentences)))
    
        # Finally, we return the k most similar sentences.
        return [sentences[i] for i in idx.tolist()]
        

        

And we can test our hybrid (sparse followed by dense) retriever on the same questions as before:

In [41]:
retriever = HybridRetriever()
for question in questions:
    print("Question:", question)
    print("Retrieved documents:")
    for relevant_doc in retriever.search(question):
        print("- " + relevant_doc.replace("\n", " "))
    print("===========")

Split strings:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/178246 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Question: Who is Luke Skywalker?
Retrieved documents:


...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

- Skywalker home: The Skywalker home was where the slaves Anakin Skywalker and his mother Shmi Skywalker Lars lived in Mos Espa on the desert planet Tatooine.
- In 32 BBY, she helped Anakin Skywalker return to the Jedi Temple after he helped her son repair their malfunctioning nanny droid.
- Tusken Raider camp (Anakin Skywalker): A Tusken Raider camp was located in the deserts of the Outer Rim planet Tatooine prior to the Clone Wars.
- It was destroyed by Anakin Skywalker out of revenge for the death of his mother, Shmi Skywalker Lars.
- During the Invasion of Naboo, Mos Espa was also home to a number of slaves, including Anakin Skywalker and his mother, Shmi.
Question: Which planet did Darth Sidious represent as senator?
Retrieved documents:


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

- The Sith Lord Darth Sidious dispatched his Sith apprentice, Darth Maul, to kill Cion and his associates after the Devaronian threatened to blackmail Senator Sheev Palpatine&mdash;Sidious's public persona&mdash;with information on his underworld dealings.
- Senator Pax Teem of the Gran Protectorate was murdered in the embassy by Darth Plagueis' apprentice, Darth Sidious, thus removing an opponent to the Sith Lords.
- Cion discovered underworld dealings of Senator Sheev Palpatine&mdash;in reality, the Sith Lord Darth Sidious, who sent his Sith apprentice, Darth Maul, to kill the Devaronian.
- Darth Sidious' shuttle: Darth Sidious' shuttle was a Republic executive shuttle used as the personal shuttle of the Sith Lord Darth Sidious in the later stages of the Clone Wars.
- During the Republic Era, between around 40 BBY and 32 BBY,The events of Darth Maul &ndash; Black, White & Red 3 take place during Darth Maul's time as the Sith apprentice of Darth Sidious, which Star Wars: Timelines dat

--------------------
### My explanation

Compared to BM25, the results are much shorter, and for some questions the most useful sentence is at the top. For example, for Kashyyyk both "forest planet" and "discovered by the Czerka Corporation" are in the first two sentences.

But I see two problems:

1. After splitting, only the first sentence of each document contains the title. The other sentences often start with "It", "They" or "He", so we lose the information about *who* the sentence is about. For example, "It was destroyed by Anakin Skywalker..." or "They adopted a Neimoidian known as Gunray...".
2. The dense step can only re-rank what BM25 already found. For the Darth Sidious question, BM25 never retrieved the Palpatine document, so no sentence mentions Naboo, and the encoder cannot fix that. The error of the first step just propagates to the second one.

--------------------

## Putting it all together

Now that we have a functioning retriever model, we can connect it to the generative language model employed to produce the responses.

**Task 4** (4 points): Implement the `RetrievalAugmentedResponseGenerator`. Given an initial question, the method should first retrieve relevant sentences using the `HybridRetriever` we have just developed. Then, it should expand the initial prompt using the provided template (you are of course free to edit or adapt it as you see fit). This expanded prompt should then be tokenized and fed as input to the LLM in the same way as before.

In [ ]:
# # Remove the old variables
# import gc
# del agent  
# del retriever
# gc.collect()
# torch.cuda.empty_cache()

--------------------
### My explanation

I didn't write the model and retriever code again. The RAG class just contains a `BasicResponseGenerator` and a `HybridRetriever`, so the model loading, chat template and output normalization are the same as in Task 1.

In `get_response`:

1. I retrieve the k most relevant sentences for the query.
2. I put them in the prompt template, joined with `"\n-"`, so each fact appears as a separate bullet point in the prompt.
3. I send this expanded prompt to the generator, which returns only the answer.

--------------------

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROMPT_TEMPLATE = "You are given the following facts about Star Wars:\n-{retrieved_sentences}\n\nWith the help of those facts and your knowledge of Star Wars, answer the following question in 1 or 2 sentences: '{query}'\nAnswer:"


class RetrievalAugmentedResponseGenerator:

    def __init__(
        self,
        model_name="google/gemma-3-1b-it",
        doc_filename="starwars.json",
        encoder_model="all-MiniLM-L6-v2",
        cache_dir="/fp/projects01/ec403/hf_models",
        device = device,
        rule_prompt: str = None
    ):
        """Loads the tokenizer, pretrained causal LM for the given model, along with the
        hybrid sparse-dense retriever model populated with the documents in doc_filename.
        """

        self.rule_prompt = rule_prompt
        self.generator = BasicResponseGenerator(model_name=model_name, device=device)
        self.retriever = HybridRetriever(filename=doc_filename, encoder_model=encoder_model)

    def get_response(self, query: str, max_length: int = 50, k=5) -> str:
        """Given a prompt, retrieve k relevant sentences, generate a response (of a maximum
        max_length tokens) and return it.
        Only the response should be returned, not the text of the prompt itself
        """

        retrieved_sentences = self.retriever.search(query, k=k)
        prompt = PROMPT_TEMPLATE.format(retrieved_sentences="\n- ".join(retrieved_sentences), query=query)
        return self.generator.get_response(prompt, max_length=max_length, rule_prompt=self.rule_prompt)

agent = RetrievalAugmentedResponseGenerator()

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Split strings:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/178246 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The last step is to test our system end-to-end:

In [8]:
for question in questions:
    print("Question:", question)
    print("Answer:", agent.get_response(question))
    print("-------")

Question: Who is Luke Skywalker?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Luke Skywalker is a prominent character in the Star Wars universe, known for his detective work, piloting skills, and being a skilled lightsaber craftsman.
-------
Question: Where is the Niima Outpost in Star Wars?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: The Skywalker home was called Mos Espa.
-------
Question: Which planet did Darth Sidious represent as senator?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Darth Sidious represented the Devaronian planet as senator.
-------


--------------------
### My explanation: 
To evaluate the answers, I used LLM to judge whether the genrated information were correct or not.

Compared to the model without retrieval, some answers are clearly better:

- **Niima Outpost:** now it says Jakku (correct), before it invented a planet "Nima".
- **Kashyyyk:** now it says the Czerka Corporation discovered it, before it said the Rebel Alliance.
- **Condlurans:** it no longer invents the planet "Xylos", and it gives Briff. But it missed Freck, even though he was in the same retrieved sentence.

But some answers are still wrong, and in most cases the reason is the retrieved context:

- **Luke Skywalker:** it says he is "a skilled detective", which comes from the comic title "Luke Skywalker: Detective". The model doesn't know that this is a title and not a fact about him.
- **First Battle of Geonosis:** it says the battle happened "thirty months after the Battle of Geonosis", which comes from the Battle of Rendili sentence. So the model mixes facts from different documents.
- **Anakin Skywalker:** it answers Tatooine / "Skywalker home", but the expected answer is Mos Espa, which was in the context too.
- **Darth Sidious:** it says "the Devaronian planet", which is wrong. No retrieved sentence mentions Naboo, so the model just picked a word from the context.

So when the right fact is in the context, the small model mostly uses it correctly. When it isn't, the model still answers confidently with whatever is in the prompt. In other words, the quality of the answer depends heavily on the retrieval step.

--------------------

**Task 5** (4 points): If you have implemented your model correctly, the system should answer correctly to at least a few questions. But it is still far from perfect, and some of the answers are flat-out wrong. Suggest 2-3 ways one could improve the current system and get even better answers. You don't need to implement anything, simply flesh out a few ideas you believe are worth trying out.

_(of course, it is even better if you actually try to implement those ideas and evaluate their influence on the quality of the system responses!)_

--------------------
### My explanation

My Suggestions:

1. Better first-stage retrieval: I think the main weakness of this pipeline is the first BM25 step. As I explained above, BM25 scores each word on its own, so if it doesn't find the right document, the dense step can't fix it. For example, for the Darth Sidious question, the Palpatine document never reached the second step, so no sentence mentioned Naboo.
   To solve this, we can embed the whole corpus once with the sentence encoder and store the vectors in a vector database. At answering time, we only embed the query and search the vector DB, so it is also fast. The best approach would be to combine both retrievers, for example with reciprocal rank fusion:

   $$\text{score}(d) = \frac{1}{60 + \text{rank}_{BM25}(d)} + \frac{1}{60 + \text{rank}_{dense}(d)}$$

   In this way, BM25 still helps with rare names like "Condluran" or "Niima", and the dense part helps when the question needs the meaning, like "which planet did he represent as senator".

2. Keep the document title with each sentence, and retrieve more candidates. After splitting into sentences, only the first sentence keeps the title, and the others start with "It" or "They". So I would add the title in front of each sentence.

3. Better prompt and a stronger model: Right now, when the right fact is not in the context, the model still answers confidently (e.g., "the Devaronian planet"). In the prompt, we can tell the model to answer *only* based on the given facts and to say "I don't know" if the answer is not there. This doesn't make the answer correct, but at least it avoids confident wrong answers. Also, a bigger model like gemma-3-4b-it would be better at choosing the right fact from the context, for example picking Mos Espa instead of only Tatooine in the Anakin question.

--------------------

--------------------
### My explanation

In the following I try to apply the second and third suggestions.

I remove the comments the previous classes and just mention the changes in the following implementation.

--------------------

In [23]:
# The second suggestion

class HybridRetriever(BM25Retriever):

    def __init__(
        self,
        filename="starwars.json",
        encoder_model="all-MiniLM-L6-v2",
        cache_dir="/fp/projects01/ec403/hf_models",
    ):

        BM25Retriever.__init__(self, filename)
        self.encoder = sentence_transformers.SentenceTransformer(
            encoder_model, 
        )

    def search(self, query: str, k: int = 5) -> List[str]:

        docs = BM25Retriever.search(self, query, k)

        # In BM25, we retrieve documents, but we want to retrieve sentences along with adding the title of the document to the sentence.
        # So after retrieving the sentences of each document, we add the title of the document to each sentence.
        sentences = list()
        for doc in docs:
            sep = doc.split(':')
            title = sep[0].strip()
            body = ":".join(sep[1:]).strip()
            for sentence in nltk.sent_tokenize(body):
                sentences.append(f"{title} : {sentence}")
            
        query_emb = self.encoder.encode_query(query, convert_to_tensor=True)
        sentences_emb = self.encoder.encode_document(sentences, convert_to_tensor=True)

        similarities = self.encoder.similarity(query_emb, sentences_emb)[0]
        scores, idx = torch.topk(similarities, k=min(k, len(sentences)))
    
        # Finally, we return the k most similar sentences.
        return [sentences[i] for i in idx.tolist()]

In [24]:
agent = RetrievalAugmentedResponseGenerator()

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Split strings:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/178246 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
for question in questions:
    print("Question:", question)
    print("Answer:", agent.get_response(question))
    print("-------")

Question: Who is Luke Skywalker?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Luke Skywalker is a character from Star Wars, known for being a detective, an operative with the Alliance, and the creator of his own lightsaber.
-------
Question: Where is the Niima Outpost in Star Wars?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: The Skywalker home was located on Tatooine, a desert planet where Anakin and his mother lived.
-------
Question: Which planet did Darth Sidious represent as senator?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Darth Sidious represented the Gran Protectorate Embassy on Mandalore.
-------


--------------------
### My explanation

Adding the title to each sentence did not clearly improve the answers.

--------------------

In [26]:
# The third suggestion 

#Change the prompt template to include the retrieved sentences as facts, and to ask the model to answer the question using only those facts.
PROMPT_TEMPLATE = "You are given the following facts about Star Wars:\n-{retrieved_sentences}\n\nWith just the help of those facts, answer the following question in 1 or 2 sentences: '{query}'\nAnswer:"
RAG_RULE_PROMPT = (
    "Answer the Star Wars question using only the given facts. "
    "If the facts don't contain the answer, say: I don't know. "
)
agent = RetrievalAugmentedResponseGenerator(rule_prompt=RAG_RULE_PROMPT)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Split strings:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/178246 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/178246 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
for question in questions:
    print("Question:", question)
    print("Answer:", agent.get_response(question))
    print("-------")

Question: Who is Luke Skywalker?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Luke Skywalker was a detective, operative, and X-Wing Mech pilot, reflecting his role as a key figure in the Star Wars universe.
-------
Question: Where is the Niima Outpost in Star Wars?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: The Skywalker home was where Anakin Skywalker and his mother, Shmi Skywalker Lars, lived on the desert planet Tatooine.
-------
Question: Which planet did Darth Sidious represent as senator?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Darth Sidious represented the Gran Protectorate Embassy as a senator.
-------


--------------------
### My explanation
Now correct: Geonosis (22 BBY, between the Confederacy and the Republic, Jango Fett's death). The model no longer says "published in 2022", so the "only the given facts" instruction helped.


Conclusion: 
The prompt change helped more than the retrieval change, but neither fixes the cases where the right document is never retrieved (Sidious). That needs a better first-stage retriever (suggestion 1).

--------------------